In [1]:
# import psycopg2
import pandas as pd
import seaborn as sns
import numpy as np
import setting_for_sda.config as conf
from setting_for_sda.constants import CONSTANTS
import lib.database.DBConn as db_conn
import json

In [2]:
# lang='python'
lang='python'
year_range = '22to24'

In [3]:
db = db_conn.DBConn()

In [4]:
lang_tag_dict = {'python' : 'python',
                'cpp': 'c++',
                'java':'java',
                'vba':'vba'
                }

In [5]:
monthly_timestamps = CONSTANTS.year_range[year_range]["monthly_timestamps"]

In [6]:
for idx in range(len(monthly_timestamps)-1):
    st_dt   = monthly_timestamps[idx].replace('.','-')
    end_dt  = monthly_timestamps[idx+1].replace('.','-')


    sql = """select p.id, p.creationdate, p.title, p.tags, p2.body from posts p , postsbody p2 where p.id = p2.id and p.posttypeid = '1' and p.tags like %s and p.creationdate >=  %s and p.creationdate < %s """

    with db.cursor() as cur:
        cur.execute(    sql,
                        (f"%<{lang}>%", st_dt, end_dt)
                   )
        
        rows = cur.fetchall()   
                
    dict_q = [{ 'id' : row[0], 
                'creationdate' : row[1].isoformat(),
                'title' : row[2],
                'tags' : row[3],
                'body' : row[4]
            } for row in rows]
    # json_str = json.dumps(dict_q, default=str, ensure_ascii=False, indent=2)

    with open(f"{CONSTANTS.data_root_dir}/data/snapshot2/questions/{lang}/{year_range}/{idx}.json", "w", encoding="utf-8") as f:
        json.dump(dict_q, f, ensure_ascii=False, indent=2)

